In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, sys, textwrap
import numpy as np
import pandas as pd

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/day14-detector-blindspot')
DATA = WORK / 'cbs-data'
OUT = WORK / 'results'
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Find the attached 20K dataset by required files and row count.
data_candidates = []
for feature_path in INPUT.rglob('features.csv'):
    data_root = feature_path.parent
    series_path = data_root / 'series.npy'
    run_path = data_root / 'scores' / 'run.json'
    if not (series_path.is_file() and run_path.is_file()):
        continue
    try:
        ids = pd.read_csv(feature_path, usecols=['source_id'], dtype={'source_id': 'string'})['source_id']
        n_rows = len(ids)
        n_sources = int(ids.nunique(dropna=True))
        if n_rows >= 19_000 and n_sources >= 10_000:
            data_candidates.append((n_rows, n_sources, data_root, feature_path, series_path, run_path))
    except Exception as exc:
        print('Skipping unreadable candidate:', feature_path, type(exc).__name__, str(exc)[:160])
if not data_candidates:
    raise FileNotFoundError('Attach the successful 20K Day 10 output containing cbs-data/features.csv, series.npy, and scores/run.json.')
N_ROWS, N_SOURCES, SOURCE_DATA, SOURCE_FEATURES, SOURCE_SERIES, SOURCE_RUN = max(data_candidates, key=lambda item: (item[0], item[1], str(item[2])))
print('[PASS] Selected 20K data:', SOURCE_DATA)
print(f'[PASS] Feature rows={N_ROWS:,}; unique source IDs={N_SOURCES:,}')

# Find the attached project source that contains the injection implementation.
project_candidates = []
for injection_path in INPUT.rglob('injections.py'):
    if injection_path.parent.name == 'cosmic_black_swan' and injection_path.parent.parent.name == 'src':
        project_root = injection_path.parent.parent.parent
        if (project_root / 'pyproject.toml').is_file():
            project_candidates.append((project_root, injection_path))
if not project_candidates:
    raise FileNotFoundError('Attach Cosmic Swan 4 source with pyproject.toml and src/cosmic_black_swan/injections.py.')
PROJECT, INJECTION_SOURCE = sorted(project_candidates, key=lambda item: str(item[0]))[0]
sys.path.insert(0, str(PROJECT / 'src'))
from cosmic_black_swan.injections import injection_recovery
print('[PASS] Project source:', PROJECT)

# Stage only the writable files required by injection_recovery.
(DATA / 'scores').mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE_FEATURES, DATA / 'features.csv')
shutil.copy2(SOURCE_SERIES, DATA / 'series.npy')
shutil.copy2(SOURCE_RUN, DATA / 'scores' / 'run.json')
staged = pd.read_csv(DATA / 'features.csv', dtype={'source_id': 'string'})
series = np.load(DATA / 'series.npy', mmap_mode='r')
required = {'source_id', 'sector', 'camera', 'ccd'}
missing = sorted(required - set(staged.columns))
if missing:
    raise ValueError(f'20K feature file is missing required columns: {missing}')
if len(staged) != len(series):
    raise ValueError(f'Feature/series row mismatch: {len(staged)} vs {len(series)}')
if staged[['sector', 'camera', 'ccd']].isna().any().any():
    raise ValueError('Detector location columns contain missing values; cannot make a detector-grouped map.')
detector_locations = int(staged[['sector', 'camera', 'ccd']].drop_duplicates().shape[0])
print('[PASS] Staged aligned data:', len(staged), 'rows; series shape:', series.shape, '; detector locations:', detector_locations)
print('[PASS] Writable run directory:', DATA)

In [ ]:
# Run all eligible held-out sources. The function caps 10,000 to the test-source count.
AMPLITUDES = (1.0, 2.0, 4.0, 8.0)
result = injection_recovery(DATA, sample_size=10_000, amplitudes_sigma=AMPLITUDES, seed=20261116)
raw_path = DATA / 'scores' / 'injection_recovery_rows.csv'
raw = pd.read_csv(raw_path)
group_cols = ['sector', 'camera', 'ccd', 'injection_family', 'amplitude_sigma']
mapped = raw.groupby(group_cols, dropna=False).agg(
    injections=('recovered_at_frozen_1pct_threshold', 'size'),
    recovered=('recovered_at_frozen_1pct_threshold', 'sum'),
    recall=('recovered_at_frozen_1pct_threshold', 'mean'),
).reset_index()

# Wilson 95% binomial interval for each detector/family/amplitude cell.
z = 1.959963984540054
n = mapped["injections"].to_numpy(dtype=float)
p = mapped["recall"].to_numpy(dtype=float)
denom = 1.0 + (z * z / n)
center = (p + z * z / (2.0 * n)) / denom
half = z * np.sqrt((p * (1.0 - p) / n) + (z * z / (4.0 * n * n))) / denom
mapped["recall_wilson95_low"] = np.clip(center - half, 0.0, 1.0)
mapped["recall_wilson95_high"] = np.clip(center + half, 0.0, 1.0)
mapped["frozen_threshold_score"] = float(result["threshold_score"])
mapped["interpretation"] = "synthetic feature-sequence injections on held-out TESS backgrounds; not full pixel-level instrument simulation"
map_path = OUT / 'day14_detector_blindspot_map.csv'
mapped.to_csv(map_path, index=False)

# Report recovery averaged over the held-out detector groups for each signal family and amplitude.
overall = raw.groupby(['injection_family', 'amplitude_sigma']).agg(injections=('recovered_at_frozen_1pct_threshold', 'size'), recall=('recovered_at_frozen_1pct_threshold', 'mean')).reset_index()
n_all = overall["injections"].to_numpy(dtype=float)
p_all = overall["recall"].to_numpy(dtype=float)
d_all = 1.0 + z * z / n_all
c_all = (p_all + z * z / (2.0 * n_all)) / d_all
h_all = z * np.sqrt((p_all * (1.0 - p_all) / n_all) + (z * z / (4.0 * n_all * n_all))) / d_all
overall["recall_wilson95_low"] = np.clip(c_all - h_all, 0.0, 1.0)
overall["recall_wilson95_high"] = np.clip(c_all + h_all, 0.0, 1.0)
overall_path = OUT / 'day14_detector_blindspot_overall.csv'
overall.to_csv(overall_path, index=False)

print("[PASS] Injection run finished")
print("Background sources sampled:", result["background_sources_sampled"])
print("Unique held-out test sources:", result["test_unique_sources"])
print("Injection rows:", len(raw))
print("Detector/family/amplitude bins:", len(mapped))
print("Frozen 1% calibration threshold:", result["threshold_score"])
display(overall.sort_values(["injection_family", "amplitude_sigma"]))
display(mapped.sort_values("recall").head(15))

In [ ]:
import matplotlib.pyplot as plt

# Overall sensitivity curves by injected morphology.
fig, ax = plt.subplots(figsize=(9, 5.5))
for family, part in overall.groupby("injection_family"):
    part = part.sort_values("amplitude_sigma")
    ax.plot(part["amplitude_sigma"], part["recall"], marker="o", label=family)
    ax.fill_between(part["amplitude_sigma"].to_numpy(dtype=float), part["recall_wilson95_low"].to_numpy(dtype=float), part["recall_wilson95_high"].to_numpy(dtype=float), alpha=0.15)
ax.set(xlabel="Injected amplitude (sigma)", ylabel="Recovery at frozen 1% threshold", ylim=(0, 1), title="Day 14 synthetic injection sensitivity")
ax.grid(alpha=0.25)
ax.legend(title="Injection family")
fig.tight_layout()
overall_plot = OUT / 'day14_detector_blindspot_overall.png'
fig.savefig(overall_plot, dpi=160)
plt.show()

# Show the lowest-recovery detector cells with at least 10 injections.
worst = mapped[mapped["injections"] >= 10].sort_values("recall").head(25).copy()
if len(worst):
    labels = [f"S{int(r.sector)} C{int(r.camera)} CCD{int(r.ccd)} | {r.injection_family} | {r.amplitude_sigma:g}σ (n={int(r.injections)})" for r in worst.itertuples()]
    y = np.arange(len(worst))
    left = worst["recall"].to_numpy(dtype=float) - worst["recall_wilson95_low"].to_numpy(dtype=float)
    right = worst["recall_wilson95_high"].to_numpy(dtype=float) - worst["recall"].to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(12, max(6, 0.34 * len(worst))))
    ax.barh(y, worst["recall"], color="#5078a0")
    ax.errorbar(worst["recall"], y, xerr=np.vstack([left, right]), fmt="none", ecolor="black", capsize=2)
    ax.set_yticks(y, labels)
    ax.invert_yaxis()
    ax.set(xlim=(0, 1), xlabel="Recovery at frozen 1% threshold (Wilson 95% CI)", title="Lowest-recovery detector/signal cells (n ≥ 10)")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    worst_plot = OUT / 'day14_detector_blindspot_low_recovery_bins.png'
    fig.savefig(worst_plot, dpi=160)
    plt.show()
else:
    worst_plot = None
    print("No detector cells reached the minimum count of 10; report the CSV counts and avoid ranking sparse bins.")

summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_data_root": str(SOURCE_DATA),
    "feature_rows": int(len(staged)),
    "unique_source_ids": int(staged["source_id"].nunique()),
    "detector_locations_in_input": int(staged[["sector", "camera", "ccd"]].drop_duplicates().shape[0]),
    "background_sources_sampled": int(result["background_sources_sampled"]),
    "train_unique_sources": int(result["train_unique_sources"]),
    "calibration_unique_sources": int(result["calibration_unique_sources"]),
    "heldout_unique_sources": int(result["test_unique_sources"]),
    "injection_rows": int(len(raw)),
    "injection_families": list(result["injection_families"]),
    "amplitudes_sigma": list(result["amplitudes_sigma"]),
    "frozen_calibration_quantile": float(result["frozen_threshold_baseline_quantile"]),
    "frozen_threshold_score": float(result["threshold_score"]),
    "detector_bins": int(len(mapped)),
    "overall_recovery": json.loads(overall.to_json(orient="records")),
    "scope": "Synthetic feature-sequence injections on source-disjoint held-out TESS backgrounds. Only detector locations present in this checkpoint are covered; this is not full pixel-level instrument simulation or an astrophysical discovery validation.",
    "outputs": [str(map_path), str(overall_path), str(overall_plot)] + ([str(worst_plot)] if worst_plot else []),
}
summary_path = OUT / 'day14_detector_blindspot_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

report_rows = int(len(staged))
report_sources = int(staged["source_id"].nunique())
report_tested = int(result["background_sources_sampled"])
report_test_total = int(result["test_unique_sources"])
report_locations = int(staged[["sector", "camera", "ccd"]].drop_duplicates().shape[0])
report_threshold = float(result["threshold_score"])
report_lines = [
    "# Day 14 detector blind-spot sensitivity map",
    "",
    f"- TESS feature rows: {report_rows:,}; unique sources: {report_sources:,}.",
    f"- Held-out sources tested: {report_tested:,} of {report_test_total:,} available.",
    f"- Injection tests: {len(raw):,}; detector/family/amplitude bins: {len(mapped):,}.",
    f"- Detector locations represented: {report_locations:,}.",
    f"- Frozen anomaly threshold: calibration-source 99th percentile (score {report_threshold:.6g}).",
    "",
    "## Overall recovery by signal family and amplitude",
    "",
    overall.to_string(index=False),
    "",
    "## Limits",
    "",
    "Injections were applied to saved standardized feature sequences, then morphology features and the frozen Isolation Forest score were evaluated. The map covers only sector/camera/CCD combinations represented by the attached data and the four tested signal families at the four selected amplitudes. It is not full pixel-level detector simulation, does not establish sensitivity to untested signals, and is not real-sky FDR or astrophysical discovery validation.",
]
(OUT / 'day14_detector_blindspot_report.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print('Saved Day 14 artifacts under:', OUT)
for path in sorted(OUT.iterdir()):
    print(f"{path.name}: {path.stat().st_size / (1024**2):.2f} MiB")